# Fine-tune on Google Colab

One-click training: **Runtime → Change runtime type → A100**, then **Runtime → Run all**.

| GPU | VRAM | Max Model | Batch | Seq |
|-----|------|-----------|-------|-----|
| T4 | 16 GB | 7B-4bit | 1 | 2048 |
| L4 | 24 GB | 14B-4bit | 1 | 2048 |
| A100 40GB | 40 GB | 32B-4bit | 1 | 4096 |
| A100 80GB | 80 GB | 32B-4bit | 4 | 4096 |

In [ ]:
#@title Setup { display-mode: "form" }
#@markdown Clone repo, install CUDA dependencies, verify GPU.

# — Clone —
import subprocess, os
REPO = "https://github.com/piotrjanik/ai-tuner.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
if not os.path.exists("ai-tuner"):
    !git clone --depth 1 --branch {BRANCH} {REPO}
%cd ai-tuner

# — Install core deps —
!pip install -q -r src/requirements-cuda.txt

# — Flash Attention (optional, speeds up training on Ampere+ GPUs) —
# Install pre-built wheel only; skip if unavailable rather than compiling from source (~30 min).
!pip install -q flash-attn --no-build-isolation --no-compile 2>/dev/null || echo "⚠ flash-attn not available as wheel — training will use eager attention instead"

# — GPU check —
import torch
assert torch.cuda.is_available(), "No GPU! Go to Runtime → Change runtime type → GPU"
props = torch.cuda.get_device_properties(0)
print(f"\n✓ GPU: {props.name} ({props.total_memory / 1e9:.0f} GB)")
print(f"✓ Flash Attention 2: {'yes' if props.major >= 8 else 'no (eager fallback)'}")

In [ ]:
#@title Prepare data & train { display-mode: "form" }
#@markdown Prepares training data from config.yaml sources, then runs QLoRA training.
#@markdown Auto-tune picks batch size, seq length, and grad checkpoint based on your GPU.

!python src/data/prepare_data.py --config config.yaml
!python src/train/train_cuda.py --config config.yaml

In [ ]:
#@title Prepare data & train { display-mode: "form" }
#@markdown Prepares training data from config.yaml sources, then runs QLoRA training.
#@markdown Auto-tune picks batch size, seq length, and grad checkpoint based on your GPU.

!python src/cli.py data
!python src/cli.py train --cuda

In [ ]:
# (Optional) Push adapters to HuggingFace Hub
# !python src/cli.py push